# Apache Spark Data Processing

## Step 1: Import Required Libraries

In this step, we import the required PySpark modules for creating a Spark session,
performing DataFrame operations, handling data types, and applying aggregation functions.

In [1]:
# Import SparkSession to create the entry point of the Spark application
from pyspark.sql import SparkSession

# Import commonly used SQL functions
from pyspark.sql.functions import (
    col,
    count,
    sum,
    avg,
    min,
    max,
    when,
    to_date,
    desc
)

# Import data types (used later for schema modifications if required)
from pyspark.sql.types import *

## Step 2: Create a Spark Session

A SparkSession is the entry point for working with DataFrames and Spark SQL.
Every Spark application starts by creating a Spark session.

In [2]:
# Create a SparkSession
# appName() gives a name to the Spark application

spark = (
    SparkSession.builder
    .appName("Week5_Spark_Data_Processing")
    .getOrCreate()
)

# Display confirmation
print("Spark Session Created Successfully!")

Spark Session Created Successfully!


## Step 3: Load the Dataset

The dataset is stored in the `data` folder.

We use:
- header=True → First row contains column names
- inferSchema=True → Spark automatically detects data types

In [3]:
# Load the CSV dataset with proper parsing options

df = (
    spark.read
    .option("header", "true")
    .option("inferSchema", "true")
    .option("multiLine", "true")
    .option("quote", '"')
    .option("escape", '"')
    .csv("../data/Sample - Superstore.csv")
)

print("Dataset Loaded Successfully!")

Dataset Loaded Successfully!


## Step 4: Explore the Dataset

Before performing any transformations, it is important to understand the dataset.

We will:
- Display sample records
- View the schema
- Count rows and columns

In [4]:
# Display the first five rows
df.show(5)

+------+--------------+----------+----------+--------------+-----------+---------------+---------+-------------+---------------+----------+-----------+------+---------------+---------------+------------+--------------------+--------+--------+--------+--------+
|Row ID|      Order ID|Order Date| Ship Date|     Ship Mode|Customer ID|  Customer Name|  Segment|      Country|           City|     State|Postal Code|Region|     Product ID|       Category|Sub-Category|        Product Name|   Sales|Quantity|Discount|  Profit|
+------+--------------+----------+----------+--------------+-----------+---------------+---------+-------------+---------------+----------+-----------+------+---------------+---------------+------------+--------------------+--------+--------+--------+--------+
|     1|CA-2016-152156| 11/8/2016|11/11/2016|  Second Class|   CG-12520|    Claire Gute| Consumer|United States|      Henderson|  Kentucky|      42420| South|FUR-BO-10001798|      Furniture|   Bookcases|Bush Somerset 

In [5]:
# Print the schema of the DataFrame
df.printSchema()

root
 |-- Row ID: integer (nullable = true)
 |-- Order ID: string (nullable = true)
 |-- Order Date: string (nullable = true)
 |-- Ship Date: string (nullable = true)
 |-- Ship Mode: string (nullable = true)
 |-- Customer ID: string (nullable = true)
 |-- Customer Name: string (nullable = true)
 |-- Segment: string (nullable = true)
 |-- Country: string (nullable = true)
 |-- City: string (nullable = true)
 |-- State: string (nullable = true)
 |-- Postal Code: integer (nullable = true)
 |-- Region: string (nullable = true)
 |-- Product ID: string (nullable = true)
 |-- Category: string (nullable = true)
 |-- Sub-Category: string (nullable = true)
 |-- Product Name: string (nullable = true)
 |-- Sales: double (nullable = true)
 |-- Quantity: integer (nullable = true)
 |-- Discount: double (nullable = true)
 |-- Profit: double (nullable = true)



In [6]:
# Count the total number of rows
print("Total Rows:", df.count())

# Count the total number of columns
print("Total Columns:", len(df.columns))

Total Rows: 9994
Total Columns: 21


In [7]:
# Display all column names
print(df.columns)

['Row ID', 'Order ID', 'Order Date', 'Ship Date', 'Ship Mode', 'Customer ID', 'Customer Name', 'Segment', 'Country', 'City', 'State', 'Postal Code', 'Region', 'Product ID', 'Category', 'Sub-Category', 'Product Name', 'Sales', 'Quantity', 'Discount', 'Profit']


## Step 5: Data Cleaning

Data cleaning is an essential step before performing any analysis.

In this section, we will:

- Remove duplicate records.
- Check for missing (null) values.
- Handle missing values if present.

This ensures that the dataset is clean and suitable for further transformations and analysis.

In [8]:
# Count the total number of records before removing duplicates
rows_before = df.count()

# Remove duplicate rows from the DataFrame
df = df.dropDuplicates()

# Count the total number of records after removing duplicates
rows_after = df.count()

# Display the results
print(f"Rows before removing duplicates : {rows_before}")
print(f"Rows after removing duplicates  : {rows_after}")
print(f"Duplicate rows removed          : {rows_before - rows_after}")

Rows before removing duplicates : 9994
Rows after removing duplicates  : 9994
Duplicate rows removed          : 0


In [9]:
# Import required functions
from pyspark.sql.functions import col, when, count

# Count the number of null values in each column
null_counts = df.select([
    count(when(col(column).isNull(), column)).alias(column)
    for column in df.columns
])

# Display null counts
null_counts.show()

+------+--------+----------+---------+---------+-----------+-------------+-------+-------+----+-----+-----------+------+----------+--------+------------+------------+-----+--------+--------+------+
|Row ID|Order ID|Order Date|Ship Date|Ship Mode|Customer ID|Customer Name|Segment|Country|City|State|Postal Code|Region|Product ID|Category|Sub-Category|Product Name|Sales|Quantity|Discount|Profit|
+------+--------+----------+---------+---------+-----------+-------------+-------+-------+----+-----+-----------+------+----------+--------+------------+------------+-----+--------+--------+------+
|     0|       0|         0|        0|        0|          0|            0|      0|      0|   0|    0|          0|     0|         0|       0|           0|           0|    0|       0|       0|     0|
+------+--------+----------+---------+---------+-----------+-------------+-------+-------+----+-----+-----------+------+----------+--------+------------+------------+-----+--------+--------+------+



In [10]:
# Fill missing values in numeric columns
df = df.na.fill({
    "Sales": "0",
    "Quantity": "0",
    "Discount": "0",
    "Profit": 0
})

# Fill missing values in string columns
df = df.na.fill({
    "Category": "Unknown",
    "Region": "Unknown"
})

print("Missing values handled successfully.")

Missing values handled successfully.


In [11]:
# Verify that missing values have been handled
df.select([
    count(when(col(column).isNull(), column)).alias(column)
    for column in df.columns
]).show()

+------+--------+----------+---------+---------+-----------+-------------+-------+-------+----+-----+-----------+------+----------+--------+------------+------------+-----+--------+--------+------+
|Row ID|Order ID|Order Date|Ship Date|Ship Mode|Customer ID|Customer Name|Segment|Country|City|State|Postal Code|Region|Product ID|Category|Sub-Category|Product Name|Sales|Quantity|Discount|Profit|
+------+--------+----------+---------+---------+-----------+-------------+-------+-------+----+-----+-----------+------+----------+--------+------------+------------+-----+--------+--------+------+
|     0|       0|         0|        0|        0|          0|            0|      0|      0|   0|    0|          0|     0|         0|       0|           0|           0|    0|       0|       0|     0|
+------+--------+----------+---------+---------+-----------+-------------+-------+-------+----+-----+-----------+------+----------+--------+------------+------------+-----+--------+--------+------+



## Step 6: Data Transformation

After cleaning the dataset, the next step is to transform the data into a suitable format for analysis.

In this section, we will:

- Rename column names for better readability.
- Convert data types to their appropriate formats.
- Convert date columns to `DateType`.
- Verify the updated schema.

### Rename Columns

Some column names contain spaces, making them inconvenient to reference in Spark queries.

We rename them using underscores for better readability.

In [12]:
# Rename columns containing spaces

df = (
    df.withColumnRenamed("Order Date", "Order_Date")
      .withColumnRenamed("Ship Date", "Ship_Date")
      .withColumnRenamed("Ship Mode", "Ship_Mode")
      .withColumnRenamed("Customer ID", "Customer_ID")
      .withColumnRenamed("Customer Name", "Customer_Name")
      .withColumnRenamed("Postal Code", "Postal_Code")
      .withColumnRenamed("Product ID", "Product_ID")
      .withColumnRenamed("Sub-Category", "Sub_Category")
      .withColumnRenamed("Product Name", "Product_Name")
)

In [13]:
from pyspark.sql.functions import to_date, col

df = (
    df.withColumn("Order_Date", to_date(col("Order_Date"), "M/d/yyyy"))
      .withColumn("Ship_Date", to_date(col("Ship_Date"), "M/d/yyyy"))
)

print("Date columns converted successfully.")

Date columns converted successfully.


### Convert Date Columns

The Order Date and Ship Date columns are currently stored as strings.

We convert them into Spark's `DateType` to enable date-based operations.

In [14]:
# Convert string dates to DateType

df = (
    df.withColumn("Order_Date", to_date(col("Order_Date"), "M/d/yyyy"))
      .withColumn("Ship_Date", to_date(col("Ship_Date"), "M/d/yyyy"))
)

print("Date columns converted successfully.")

Date columns converted successfully.


In [15]:
# Display updated schema
df.printSchema()

root
 |-- Row ID: integer (nullable = true)
 |-- Order ID: string (nullable = true)
 |-- Order_Date: date (nullable = true)
 |-- Ship_Date: date (nullable = true)
 |-- Ship_Mode: string (nullable = true)
 |-- Customer_ID: string (nullable = true)
 |-- Customer_Name: string (nullable = true)
 |-- Segment: string (nullable = true)
 |-- Country: string (nullable = true)
 |-- City: string (nullable = true)
 |-- State: string (nullable = true)
 |-- Postal_Code: integer (nullable = true)
 |-- Region: string (nullable = false)
 |-- Product_ID: string (nullable = true)
 |-- Category: string (nullable = false)
 |-- Sub_Category: string (nullable = true)
 |-- Product_Name: string (nullable = true)
 |-- Sales: double (nullable = true)
 |-- Quantity: integer (nullable = true)
 |-- Discount: double (nullable = true)
 |-- Profit: double (nullable = false)



In [16]:
# Display first five rows after transformation
df.show(5)

+------+--------------+----------+----------+--------------+-----------+----------------+---------+-------------+------------+------------+-----------+------+---------------+---------------+------------+--------------------+-------+--------+--------+----------+
|Row ID|      Order ID|Order_Date| Ship_Date|     Ship_Mode|Customer_ID|   Customer_Name|  Segment|      Country|        City|       State|Postal_Code|Region|     Product_ID|       Category|Sub_Category|        Product_Name|  Sales|Quantity|Discount|    Profit|
+------+--------------+----------+----------+--------------+-----------+----------------+---------+-------------+------------+------------+-----------+------+---------------+---------------+------------+--------------------+-------+--------+--------+----------+
|    28|US-2015-150630|2015-09-17|2015-09-21|Standard Class|   TB-21520| Tracy Blumstein| Consumer|United States|Philadelphia|Pennsylvania|      19140|  East|FUR-BO-10004834|      Furniture|   Bookcases|Riverside P

# Step 7: Data Filtering

Data filtering allows us to retrieve records that satisfy specific conditions.

The assignment mentions filtering based on age, category, and region. Since the Superstore dataset does not contain an **Age** column, we demonstrate filtering using the available attributes:

- Region
- Category
- Sales
- Profit

Filtering helps in analyzing specific subsets of the dataset for business insights.

### Filter Records by Region

In this example, we filter the dataset to display only the orders from the **West** region.

In [17]:
# Filter records where the Region is "West"

west_df = df.filter(col("Region") == "West")

# Display the total number of records in the West region
print("Total records in West Region:", west_df.count())

# Display the first five records
west_df.show(5)

Total records in West Region: 3203
+------+--------------+----------+----------+--------------+-----------+-------------------+---------+-------------+-------------+----------+-----------+------+---------------+---------------+------------+--------------------+-------+--------+--------+-------+
|Row ID|      Order ID|Order_Date| Ship_Date|     Ship_Mode|Customer_ID|      Customer_Name|  Segment|      Country|         City|     State|Postal_Code|Region|     Product_ID|       Category|Sub_Category|        Product_Name|  Sales|Quantity|Discount| Profit|
+------+--------------+----------+----------+--------------+-----------+-------------------+---------+-------------+-------------+----------+-----------+------+---------------+---------------+------------+--------------------+-------+--------+--------+-------+
|   203|CA-2014-133690|2014-08-03|2014-08-05|   First Class|   BS-11755|      Bruce Stewart| Consumer|United States|       Denver|  Colorado|      80219|  West|OFF-AP-10003622|Office

### Filter Records by Category

Here, we retrieve only the records belonging to the **Technology** category.

In [18]:
# Filter records where the Category is "Technology"

technology_df = df.filter(col("Category") == "Technology")

print("Total Technology Orders:", technology_df.count())

technology_df.show(5)

Total Technology Orders: 1847
+------+--------------+----------+----------+--------------+-----------+----------------+---------+-------------+-------------+----------+-----------+-------+---------------+----------+------------+--------------------+--------+--------+--------+---------+
|Row ID|      Order ID|Order_Date| Ship_Date|     Ship_Mode|Customer_ID|   Customer_Name|  Segment|      Country|         City|     State|Postal_Code| Region|     Product_ID|  Category|Sub_Category|        Product_Name|   Sales|Quantity|Discount|   Profit|
+------+--------------+----------+----------+--------------+-----------+----------------+---------+-------------+-------------+----------+-----------+-------+---------------+----------+------------+--------------------+--------+--------+--------+---------+
|  2935|US-2016-169040|2016-12-06|2016-12-12|Standard Class|   GT-14710|       Greg Tran| Consumer|United States|      Seattle|Washington|      98105|   West|TEC-PH-10002834|Technology|      Phones| 

### Filter Records by Sales

This example filters all orders where the sales amount is greater than **500**.

In [19]:
# Filter records with Sales greater than 500

high_sales_df = df.filter(col("Sales") > 500)

print("Orders with Sales greater than 500:", high_sales_df.count())

high_sales_df.show(5)

Orders with Sales greater than 500: 1162
+------+--------------+----------+----------+--------------+-----------+----------------+---------+-------------+------------+------------+-----------+-------+---------------+---------------+------------+--------------------+--------+--------+--------+----------+
|Row ID|      Order ID|Order_Date| Ship_Date|     Ship_Mode|Customer_ID|   Customer_Name|  Segment|      Country|        City|       State|Postal_Code| Region|     Product_ID|       Category|Sub_Category|        Product_Name|   Sales|Quantity|Discount|    Profit|
+------+--------------+----------+----------+--------------+-----------+----------------+---------+-------------+------------+------------+-----------+-------+---------------+---------------+------------+--------------------+--------+--------+--------+----------+
|    28|US-2015-150630|2015-09-17|2015-09-21|Standard Class|   TB-21520| Tracy Blumstein| Consumer|United States|Philadelphia|Pennsylvania|      19140|   East|FUR-BO-1

### Apply Multiple Filter Conditions

Spark allows multiple conditions using logical operators.

In this example, we retrieve Technology products sold in the West region.

In [20]:
# Filter Technology products sold in the West region

west_technology_df = df.filter(
    (col("Region") == "West") &
    (col("Category") == "Technology")
)

print("Technology Orders in West Region:", west_technology_df.count())

west_technology_df.show(5)

Technology Orders in West Region: 599
+------+--------------+----------+----------+--------------+-----------+-----------------+-----------+-------------+-------------+----------+-----------+------+---------------+----------+------------+--------------------+-------+--------+--------+-------+
|Row ID|      Order ID|Order_Date| Ship_Date|     Ship_Mode|Customer_ID|    Customer_Name|    Segment|      Country|         City|     State|Postal_Code|Region|     Product_ID|  Category|Sub_Category|        Product_Name|  Sales|Quantity|Discount| Profit|
+------+--------------+----------+----------+--------------+-----------+-----------------+-----------+-------------+-------------+----------+-----------+------+---------------+----------+------------+--------------------+-------+--------+--------+-------+
|  2935|US-2016-169040|2016-12-06|2016-12-12|Standard Class|   GT-14710|        Greg Tran|   Consumer|United States|      Seattle|Washington|      98105|  West|TEC-PH-10002834|Technology|      P

### Filter Profitable Orders

This example retrieves only the orders where the profit is greater than zero.

In [21]:
# Filter records with positive profit

profit_df = df.filter(col("Profit") > 0)

print("Profitable Orders:", profit_df.count())

profit_df.show(5)

Profitable Orders: 8058
+------+--------------+----------+----------+--------------+-----------+----------------+---------+-------------+--------+-----------+-----------+-------+---------------+---------------+------------+--------------------+-------+--------+--------+-------+
|Row ID|      Order ID|Order_Date| Ship_Date|     Ship_Mode|Customer_ID|   Customer_Name|  Segment|      Country|    City|      State|Postal_Code| Region|     Product_ID|       Category|Sub_Category|        Product_Name|  Sales|Quantity|Discount| Profit|
+------+--------------+----------+----------+--------------+-----------+----------------+---------+-------------+--------+-----------+-----------+-------+---------------+---------------+------------+--------------------+-------+--------+--------+-------+
|   203|CA-2014-133690|2014-08-03|2014-08-05|   First Class|   BS-11755|   Bruce Stewart| Consumer|United States|  Denver|   Colorado|      80219|   West|OFF-AP-10003622|Office Supplies|  Appliances|Bravo II Meg

# Step 8: Aggregation Functions

Aggregation functions summarize the data by performing calculations on one or more columns.

In this section, we calculate:

- Total number of records
- Total sales
- Average sales
- Minimum sales
- Maximum sales
- Profit statistics

These operations help us understand the overall characteristics of the dataset.

### Count Total Records

The `count()` function returns the total number of records in the dataset.

In [22]:
# Count the total number of records

total_records = df.count()

print("Total Records:", total_records)

Total Records: 9994


### Sales Statistics

Calculate summary statistics for the **Sales** column.

The following metrics are computed:

- Total Sales
- Average Sales
- Minimum Sale
- Maximum Sale

In [23]:
from pyspark.sql.functions import sum, avg, min, max

# Calculate summary statistics for Sales

sales_summary = df.select(
    sum("Sales").alias("Total_Sales"),
    avg("Sales").alias("Average_Sales"),
    min("Sales").alias("Minimum_Sale"),
    max("Sales").alias("Maximum_Sale")
)

sales_summary.show()

+------------------+------------------+------------+------------+
|       Total_Sales|     Average_Sales|Minimum_Sale|Maximum_Sale|
+------------------+------------------+------------+------------+
|2297200.8602999873|229.85800083049702|       0.444|    22638.48|
+------------------+------------------+------------+------------+



### Profit Statistics

Calculate summary statistics for the **Profit** column.

In [24]:
# Calculate summary statistics for Profit

profit_summary = df.select(
    sum("Profit").alias("Total_Profit"),
    avg("Profit").alias("Average_Profit"),
    min("Profit").alias("Minimum_Profit"),
    max("Profit").alias("Maximum_Profit")
)

profit_summary.show()

+-----------------+-----------------+--------------+--------------+
|     Total_Profit|   Average_Profit|Minimum_Profit|Maximum_Profit|
+-----------------+-----------------+--------------+--------------+
|286397.0216999997|28.65689630778464|     -6599.978|      8399.976|
+-----------------+-----------------+--------------+--------------+



### Quantity Statistics

Calculate summary statistics for the **Quantity** column.

In [25]:
# Calculate summary statistics for Quantity

quantity_summary = df.select(
    sum("Quantity").alias("Total_Quantity"),
    avg("Quantity").alias("Average_Quantity"),
    min("Quantity").alias("Minimum_Quantity"),
    max("Quantity").alias("Maximum_Quantity")
)

quantity_summary.show()

+--------------+-----------------+----------------+----------------+
|Total_Quantity| Average_Quantity|Minimum_Quantity|Maximum_Quantity|
+--------------+-----------------+----------------+----------------+
|         37873|3.789573744246548|               1|              14|
+--------------+-----------------+----------------+----------------+



### Multiple Aggregations

The `agg()` function allows multiple aggregation operations to be performed in a single statement.

In [26]:
from pyspark.sql.functions import count

# Perform multiple aggregations using agg()

df.agg(
    count("*").alias("Total_Orders"),
    sum("Sales").alias("Total_Sales"),
    avg("Sales").alias("Average_Sales"),
    min("Sales").alias("Minimum_Sale"),
    max("Sales").alias("Maximum_Sale")
).show()

+------------+------------------+------------------+------------+------------+
|Total_Orders|       Total_Sales|     Average_Sales|Minimum_Sale|Maximum_Sale|
+------------+------------------+------------------+------------+------------+
|        9994|2297200.8602999873|229.85800083049702|       0.444|    22638.48|
+------------+------------------+------------------+------------+------------+



# Step 9: GroupBy Operations

Grouping is used to divide the dataset into categories and calculate aggregate statistics for each group.

In this section, we analyze:

- Sales by Category
- Sales by Region
- Profit by State
- Cities with more than 100 orders

### Sales by Category

Calculate the total number of orders, total sales, and average sales for each product category.

In [27]:
# Group data by Category

category_summary = (
    df.groupBy("Category")
      .agg(
          count("*").alias("Total_Orders"),
          sum("Sales").alias("Total_Sales"),
          avg("Sales").alias("Average_Sales")
      )
      .orderBy(col("Total_Sales").desc())
)

category_summary.show()

+---------------+------------+-----------------+------------------+
|       Category|Total_Orders|      Total_Sales|     Average_Sales|
+---------------+------------+-----------------+------------------+
|     Technology|        1847|836154.0329999996| 452.7092761234432|
|      Furniture|        2121|741999.7953000005|349.83488698727035|
|Office Supplies|        6026|719047.0320000009|119.32410089611699|
+---------------+------------+-----------------+------------------+



### Sales by Region

Calculate total and average sales for each region.

In [28]:
# Group data by Region

region_summary = (
    df.groupBy("Region")
      .agg(
          sum("Sales").alias("Total_Sales"),
          avg("Sales").alias("Average_Sales")
      )
      .orderBy(col("Total_Sales").desc())
)

region_summary.show()

+-------+------------------+------------------+
| Region|       Total_Sales|     Average_Sales|
+-------+------------------+------------------+
|   West| 725457.8245000003|226.49323275054647|
|   East| 678781.2400000002|238.33610955056187|
|Central| 501239.8907999997|215.77266069737396|
|  South|391721.90500000026|241.80364506172856|
+-------+------------------+------------------+



### Profit by State

Calculate the total profit earned in each state.

In [29]:
# Group data by State

state_summary = (
    df.groupBy("State")
      .agg(
          sum("Profit").alias("Total_Profit")
      )
      .orderBy(col("Total_Profit").desc())
)

state_summary.show(10)

+----------+------------------+
|     State|      Total_Profit|
+----------+------------------+
|California| 76381.38709999992|
|  New York| 74038.54859999998|
|Washington|33402.651699999995|
|  Michigan|24463.187600000005|
|  Virginia|18597.950399999998|
|   Indiana|        18382.9363|
|   Georgia|16250.043300000001|
|  Kentucky|11199.696600000003|
| Minnesota|10823.187399999999|
|  Delaware|         9977.3748|
+----------+------------------+
only showing top 10 rows


### Cities with More Than 100 Orders

Group the dataset by city and display only those cities having more than 100 orders.

In [30]:
# Count orders by city and display cities having more than 100 orders

city_summary = (
    df.groupBy("City")
      .agg(count("*").alias("Total_Orders"))
      .filter(col("Total_Orders") > 100)
      .orderBy(col("Total_Orders").desc())
)

city_summary.show()

+-------------+------------+
|         City|Total_Orders|
+-------------+------------+
|New York City|         915|
|  Los Angeles|         747|
| Philadelphia|         537|
|San Francisco|         510|
|      Seattle|         428|
|      Houston|         377|
|      Chicago|         314|
|     Columbus|         222|
|    San Diego|         170|
|  Springfield|         163|
|       Dallas|         157|
| Jacksonville|         125|
|      Detroit|         115|
+-------------+------------+



# Step 10: Wide Transformations and Shuffle

Spark transformations are classified into two types:

### Narrow Transformations
- Data remains within the same partition.
- No data movement between partitions.
- Faster execution.
- Examples:
  - filter()
  - select()
  - withColumn()

### Wide Transformations
- Data is shuffled across partitions.
- Requires network communication.
- More expensive than narrow transformations.
- Examples:
  - groupBy()
  - join()
  - distinct()

### Shuffle

Shuffle is the process of redistributing data across partitions so that records with the same key are grouped together.

Although shuffle is expensive, it is necessary for operations like `groupBy()` and `join()`.

In [31]:
# Example of a wide transformation

region_sales = (
    df.groupBy("Region")
      .agg(sum("Sales").alias("Total_Sales"))
)

region_sales.show()

+-------+------------------+
| Region|       Total_Sales|
+-------+------------------+
|  South|391721.90500000026|
|Central| 501239.8907999997|
|   East| 678781.2400000002|
|   West| 725457.8245000003|
+-------+------------------+



# Step 11: Complete Data Processing Pipeline

In this section, we combine all the data processing steps into a single pipeline.

Pipeline Steps:

1. Remove duplicate records.
2. Handle missing values.
3. Filter valid sales records.
4. Group data by Region.
5. Calculate total and average sales.

In [32]:
from pyspark.sql.functions import sum, avg, col

# Complete Spark data processing pipeline

pipeline_df = (
    df.dropDuplicates()
      .na.fill({
          "Sales": 0,
          "Profit": 0
      })
      .filter(col("Sales") > 0)
      .groupBy("Region")
      .agg(
          sum("Sales").alias("Total_Sales"),
          avg("Sales").alias("Average_Sales")
      )
      .orderBy(col("Total_Sales").desc())
)

# Display pipeline output
pipeline_df.show()

+-------+-----------------+------------------+
| Region|      Total_Sales|     Average_Sales|
+-------+-----------------+------------------+
|   West|725457.8245000008| 226.4932327505466|
|   East|678781.2400000007|238.33610955056204|
|Central|501239.8908000003|215.77266069737422|
|  South|391721.9050000001|241.80364506172845|
+-------+-----------------+------------------+



# Step 12: Save Processed Results

Save the final processed results as a CSV file.

In [34]:
pipeline_df.coalesce(1) \
    .write \
    .mode("overwrite") \
    .option("header", True) \
    .csv("../output/final_results")